In [ ]:
# Fetch match payloads for all 15 matches, appending only those with data
import os
from typing import Optional

try:
    from dotenv import load_dotenv

    load_dotenv()
except ModuleNotFoundError:
    pass

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://fantasy.sixnationsrugby.com"


def _build_session() -> requests.Session:
    s = requests.Session()
    retries = Retry(
        total=3, backoff_factor=0.5, status_forcelist=[429, 500, 502, 503, 504]
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    return s


def fetch_match(
    match_id: int,
    language: str = "en",
    token: Optional[str] = None,
    x_access_key: Optional[str] = None,
) -> dict:
    """Fetch a match payload from the Six Nations Fantasy API.

    Provide your browser token via the SIXNATIONS_TOKEN env var (with or without leading 'Token ').
    Optionally override x-access-key via SIXNATIONS_X_ACCESS_KEY.
    """
    url = f"{BASE_URL}/v1/private/match/{match_id}?lg={language}"
    headers = {
        "accept": "application/json",
        "content-type": "application/json",
        "cache-control": "no-cache",
        "pragma": "no-cache",
    }
    if x_access_key is None:
        x_access_key = os.getenv("SIXNATIONS_X_ACCESS_KEY", "600@18.23@")
    headers["x-access-key"] = x_access_key

    if token is None:
        token = os.getenv("SIXNATIONS_TOKEN")
    if token:
        headers["authorization"] = (
            token if str(token).lower().startswith("token ") else f"Token {token}"
        )

    sess = _build_session()
    r = sess.get(url, headers=headers, allow_redirects=True, timeout=20)
    if r.status_code in (401, 403):
        raise RuntimeError(
            "Unauthorized (401/403). Ensure SIXNATIONS_TOKEN is set correctly."
        )
    r.raise_for_status()
    return r.json()


# Collect matches 1..15; only keep those with player data present
MATCH_IDS = list(range(1, 16))
all_payloads = []  # list of { 'match_id': int, 'data': dict }
for mid in MATCH_IDS:
    try:
        d = fetch_match(mid)
        m = d.get("match", {}) if isinstance(d, dict) else {}
        dom_players = m.get("joueursdom") or []
        ext_players = m.get("joueursext") or []
        if not dom_players and not ext_players:
            print(f"Match {mid}: no player data yet, skipping.")
            continue
        all_payloads.append({"match_id": mid, "data": d})
        print(f"Fetched match {mid}")
    except Exception as e:
        print(f"Match {mid}: skipping due to error -> {e}")

if not all_payloads:
    raise SystemExit("No matches returned with data. Check your token and try again.")


Fetched match 1
Match 2: no player data yet, skipping.
Match 3: no player data yet, skipping.
Match 4: no player data yet, skipping.
Match 5: no player data yet, skipping.
Match 6: no player data yet, skipping.
Match 7: no player data yet, skipping.
Match 8: no player data yet, skipping.
Match 9: no player data yet, skipping.
Match 10: no player data yet, skipping.
Match 11: no player data yet, skipping.
Match 12: no player data yet, skipping.
Match 13: no player data yet, skipping.
Match 14: no player data yet, skipping.
Match 15: no player data yet, skipping.


In [ ]:
# Transform: flatten players from each payload, compute fantasy points columns
import pandas as pd
from IPython.display import display

# Position mapping (from API position codes)
position_map = {
    "6": "Back-three",
    "7": "Centre",
    "8": "Fly-half",
    "9": "Scrum-half",
    "10": "Back-row",
    "11": "Second-row",
    "12": "Prop",
    "13": "Hooker",
}
forwards = {"Back-row", "Second-row", "Prop", "Hooker"}
backs = {"Back-three", "Centre", "Fly-half", "Scrum-half"}

# Scoring rules
scoring = {
    "Try": None,
    "Assists": 4,
    "Conversion": 2,
    "Penalty": 3,
    "DropGoal": 5,
    "DefendersBeaten": 2,
    "MetresCarried": 0.1,
    "FiftyTwentyTwo": 7,
    "KicksRecovered": 2,
    "Offloads": 2,
    "AttackingScrumWin": 1,
    "Tackles": 1,
    "BreakdownSteal": 5,
    "LineoutSteal": 7,
    "PenaltyConceded": -1,
    "PlayerOfTheMatch": 15,
    "YellowCard": -5,
    "RedCard": -8,
    "Minutes": 0,
}

# Friendly names for legend short labels
stat_name_map = {
    "Min": "Minutes",
    "T": "Try",
    "As": "Assists",
    "C": "Conversion",
    "Pen": "Penalty",
    "MC": "MetresCarried",
    "DB": "DefendersBeaten",
    "Ta": "Tackles",
    "CPen": "PenaltyConceded",
    "50-22": "FiftyTwentyTwo",
    "KR": "KicksRecovered",
    "DG": "DropGoal",
    "OF": "Offloads",
    "LS": "LineoutSteal",
    "BS": "BreakdownSteal",
    "POTM": "PlayerOfTheMatch",
    "SW": "AttackingScrumWin",
    "YC": "YellowCard",
    "RC": "RedCard",
}


def _to_int_or_zero(v) -> int:
    try:
        if v is None:
            return 0
        if isinstance(v, int):
            return int(v)
        s = str(v).strip().replace(",", "")
        return int(float(s))
    except Exception:
        return 0


def _to_float_or_zero(v) -> float:
    try:
        if v is None:
            return 0.0
        if isinstance(v, (int, float)):
            return float(v)
        s = str(v).strip().replace(",", "")
        return float(s)
    except Exception:
        return 0.0


def _pick_from_dict(d: dict, keys):
    for k in keys:
        if isinstance(d, dict) and k in d and d[k]:
            return d[k]
    return None


def extract_team_names(match_obj: dict):
    dom_name = match_obj.get("clubdom") or None
    ext_name = match_obj.get("clubext") or None
    # Fallback keys if clubdom/clubext are missing
    dom_keys = [
        "clubdom",
        "nomdom",
        "paysdom",
        "home_team",
        "homeTeam",
        "equipeDom",
        "equipe_dom",
    ]
    ext_keys = [
        "clubext",
        "nomext",
        "paysext",
        "away_team",
        "awayTeam",
        "equipeExt",
        "equipe_ext",
    ]

    def as_name(v):
        if isinstance(v, dict):
            return _pick_from_dict(
                v,
                [
                    "nom",
                    "name",
                    "libelle",
                    "label",
                    "pays",
                    "country",
                    "display_name",
                    "shortName",
                    "short_name",
                ],
            )
        return str(v) if v is not None else None

    for k in dom_keys:
        if dom_name:
            break
        v = match_obj.get(k)
        n = as_name(v)
        if n:
            dom_name = n
            break
    for k in ext_keys:
        if ext_name:
            break
        v = match_obj.get(k)
        n = as_name(v)
        if n:
            ext_name = n
            break
    return dom_name or "Home", ext_name or "Away"


def _as_name_from_maybe_dict(v):
    if isinstance(v, dict):
        for k in [
            "name",
            "nom",
            "libelle",
            "label",
            "display_name",
            "shortName",
            "short_name",
            "country",
            "pays",
        ]:
            if k in v and v[k]:
                return str(v[k])
    return str(v) if v not in (None, "") else None


def extract_player_country(
    p: dict, fallback_team: Optional[str] = None
) -> Optional[str]:
    """Derive a player's country/team label from the player payload.
    Tries several likely fields and nested dict shapes seen in the API.
    Falls back to the provided team label (home/away name) if none found.
    """
    candidates = []
    # Common top-level keys or nested objects
    for key in [
        "country",
        "pays",
        "nation",
        "team",
        "equipe",
        "selection",
        "national_team",
        "countryObj",
        "nationObj",
        "equipeObj",
    ]:
        if key in p and p[key]:
            candidates.append(p[key])
    for v in candidates:
        name = _as_name_from_maybe_dict(v)
        if name:
            return name
    return fallback_team


def flatten_players(players, team_name, legend):
    rows = []
    legend_map = {}
    for i, lg in enumerate(legend):
        short = lg.get("label_short", "")
        legend_map[i] = stat_name_map.get(short, f"crit_{i}")
    for p in players:
        pos_key = str(p.get("position", ""))
        pos_name = position_map.get(pos_key, pos_key)
        # Use match-level team labels (clubdom/clubext) for each side
        team_label = team_name
        row = {
            "id": p.get("id"),
            "name": p.get("nom"),
            "position": pos_name,
            "points_total": p.get("points", 0),
            'team': team_label,
        }
        for i, crit in enumerate(p.get("criteres", [])):
            stat_col = legend_map.get(i, f"crit_{i}")
            raw_val = crit.get("value")
            val_num = (
                _to_float_or_zero(raw_val)
                if stat_col == "MetresCarried"
                else _to_int_or_zero(raw_val)
            )
            row[stat_col] = val_num
            if stat_col == "Try":
                row["Try_points"] = (10 if pos_name in backs else 15) * int(val_num)
            elif stat_col == "MetresCarried":
                row["MetresCarried_points"] = int(val_num // 10)
            else:
                if stat_col in scoring:
                    row[f"{stat_col}_points"] = int(val_num) * scoring[stat_col]
        rows.append(row)
    return pd.DataFrame(rows)


# Build combined DataFrame across fetched matches
df_parts = []
for item in all_payloads:
    m = item["data"].get("match", {})
    legend = m.get("legende", [])
    dom_players = m.get("joueursdom") or []
    ext_players = m.get("joueursext") or []
    team_dom, team_ext = extract_team_names(m)
    dfd = (
        flatten_players(dom_players, team_dom, legend)
        if dom_players
        else pd.DataFrame()
    )
    dfe = (
        flatten_players(ext_players, team_ext, legend)
        if ext_players
        else pd.DataFrame()
    )
    dfi = (
        pd.concat([dfd, dfe], ignore_index=True)
        if not dfd.empty or not dfe.empty
        else pd.DataFrame()
    )
    if not dfi.empty:
        dfi["match_id"] = item["match_id"]
        df_parts.append(dfi)

if not df_parts:
    raise SystemExit("No usable player rows found across requested matches.")

df = pd.concat(df_parts, ignore_index=True, sort=True)

# Order columns: base + stat/value/points + extras
base_cols = ["match_id", "id", "name", "team", "position", "points_total"]
ordered_stats = [
    "Minutes",
    "Try",
    "Assists",
    "Conversion",
    "Penalty",
    "MetresCarried",
    "DefendersBeaten",
    "Tackles",
    "PenaltyConceded",
    "FiftyTwentyTwo",
    "KicksRecovered",
    "DropGoal",
    "Offloads",
    "LineoutSteal",
    "BreakdownSteal",
    "PlayerOfTheMatch",
    "AttackingScrumWin",
    "YellowCard",
    "RedCard",
]
ordered_cols = [s for s in ordered_stats if s in df.columns]
for s in ordered_stats:
    pts_col = f"{s}_points"
    if pts_col in df.columns:
        ordered_cols.append(pts_col)
extra_cols = [c for c in df.columns if c not in base_cols + ordered_cols]
df = df[base_cols + ordered_cols + extra_cols]

# Compute totals/delta
# Only coerce numeric-like columns; keep string labels like team/position intact
numeric_cols = [
    c
    for c in df.columns
    if c not in {"id", "name", "team", "position", "opponent"}
]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
point_cols = [c for c in df.columns if c.endswith("_points")]
df[point_cols] = df[point_cols].fillna(0)
df["computed_points_total"] = df[point_cols].sum(axis=1)
df["points_delta"] = df["points_total"] - df["computed_points_total"]

# Preview
display(df.head())


,match_id,id,name,team,position,points_total,Minutes,Try,Assists,Conversion,...,Offloads_points,LineoutSteal_points,BreakdownSteal_points,PlayerOfTheMatch_points,AttackingScrumWin_points,YellowCard_points,country,crit_18,computed_points_total,points_delta
0,1,1322,T. Attissogbe,France,Back-three,32,80,1,0,0,...,6,0,0,0,0,0,France,0,32,0
1,1,704,L. Bielle-Biarrey,France,Back-three,51,80,2,0,0,...,0,0,0,0,0,0,France,0,51,0
2,1,108,T. Ramos,France,Back-three,46,80,0,1,4,...,6,0,0,0,0,0,France,0,46,0
3,1,1260,N. Depoortere,France,Centre,36,80,0,0,0,...,2,0,0,0,0,0,France,0,36,0
4,1,1662,K. Gourgues,France,Centre,6,30,0,0,0,...,2,0,0,0,0,0,France,0,6,0


In [ ]:
# Persist to ./data for the dashboard (DuckDB)
import duckdb

os.makedirs("data", exist_ok=True)
db_path = "data/all_matches.duckdb"

def _qident(name: str) -> str:
    return "\"" + str(name).replace("\"", "\\\"") + "\""

con = duckdb.connect(db_path)
try:
    con.register("df", df)

    # Create table if missing
    con.execute("CREATE TABLE IF NOT EXISTS all_matches AS SELECT * FROM df LIMIT 0")

    # Ensure required keys
    required_keys = ["match_id", "id"]
    missing_keys = [k for k in required_keys if k not in df.columns]
    if missing_keys:
        raise SystemExit(f"Missing key columns in df: {missing_keys}")

    # Add new columns from df to table if needed
    existing_cols = [
        row[0]
        for row in con.execute("""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = 'main' AND table_name = 'all_matches'
            ORDER BY ordinal_position
        """).fetchall()
    ]
    df_cols = list(df.columns)

    desc = con.execute("DESCRIBE SELECT * FROM df").df()
    name_key = "column_name" if "column_name" in desc.columns else "name"
    type_key = "column_type" if "column_type" in desc.columns else "type"
    df_types = {}
    if name_key in desc.columns and type_key in desc.columns:
        df_types = dict(zip(desc[name_key], desc[type_key]))

    for col in df_cols:
        if col not in existing_cols:
            col_type = df_types.get(col, "VARCHAR")
            con.execute(f"ALTER TABLE all_matches ADD COLUMN {_qident(col)} {col_type}")
            existing_cols.append(col)

    # Align df to table schema
    select_exprs = []
    for col in existing_cols:
        if col in df_cols:
            select_exprs.append(_qident(col))
        else:
            select_exprs.append(f"NULL AS {_qident(col)}")
    con.execute(f"CREATE OR REPLACE TEMP VIEW df_aligned AS SELECT {', '.join(select_exprs)} FROM df")

    # Upsert by (match_id, id)
    con.execute("""
        DELETE FROM all_matches
        USING df_aligned
        WHERE all_matches.match_id = df_aligned.match_id
          AND all_matches.id = df_aligned.id
    """)
    con.execute("INSERT INTO all_matches SELECT * FROM df_aligned")
finally:
    con.close()

print("Saved:", db_path)


Saved: data/all_matches_20260206T134035Z.parquet and data/all_matches_20260206T134035Z.csv


/var/folders/1v/zt0x7nqs6b72__txg_sq1vwh0000gp/T/ipykernel_51802/2665142501.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
